### 📦 Offline Dependency Installation

- Installs `datasets` into a local directory (`/kaggle/working/packages`)
- Reuses Kaggle offline packages if available
- Keeps the baseline notebook style so the runtime setup stays familiar

✅ Kaggle-compatible setup for the bit_synth exact-trace CoT notebook


In [ ]:
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = (pth_file.parent / relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)
resolve_python_path(Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/"))

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets"
    ])
    print("Installed datasets from offline packages")

sys.path.append(target_dir)
resolve_python_path(Path(target_dir))


### ⚙️ Environment Setup & Libraries

- Keeps the baseline notebook environment style
- Reuses the same assistant-only training stack as the merge-LoRA baseline
- Switches only the dataset input to the bit_synth exact-trace CSV built from README-compatible boxed supervision

✅ The training plumbing stays stable while the dataset changes to bit manipulation exact-trace supervision


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile, csv, json, random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType


### 🛠️ Triton + RMSNorm Fix (Kaggle Hack)

- Preserves the baseline notebook's Kaggle-specific Triton patch
- Keeps the runtime behavior close to the original notebook

✅ Minimizes unrelated changes outside the Phase 2 dataset swap


In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")


### 📊 bit_synth Config + Kaggle Paths

**Goal:**
- Keep the proven assistant-only training path and notebook format
- Read the completed bit_synth exact-trace CSV from fixed Kaggle paths

**Data inputs:**
- Required: prebuilt bit_synth exact-trace CSV from a Kaggle dataset
- Optional: matching manifest JSON at a fixed path if uploaded together

✅ Uses Kaggle-style fixed paths and keeps the notebook in the older load-and-train shape


In [ ]:
# === Hyperparameters ===
SUBSAMPLE_SIZE = 10000
LORA_RANK = 32
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 0.75
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
SEED = 42
BOXED_INSTRUCTION = "Please put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
OUTPUT_DIR = "/kaggle/working/adapter_bit_synth_exact_trace_cot_merge_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Kaggle paths ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
DATASET_CSV = "/kaggle/input/bit-synth-exact-trace-cot-v1/bit_synth_exact_trace_cot_training_data_v1.csv"
MANIFEST_JSON = "/kaggle/input/bit-synth-exact-trace-cot-v1/bit_synth_exact_trace_cot_manifest_v1.json"
OUTPUT_COLUMNS = [
    "id", "prompt", "answer", "generated_cot", "label", "assistant_style", "source_selection_tier"
 ]


### 🧱 bit_synth Dataset Input

- Keeps the same notebook/training format as `phase1_assistant_only`
- Reads the completed bit_synth exact-trace CSV directly
- Optionally reads the matching manifest JSON for a quick sanity check

✅ No local CSV rebuild and no in-notebook trace rewriting; the notebook consumes the finished artifact directly


In [ ]:
if not os.path.exists(DATASET_CSV):
    raise FileNotFoundError(
        f"Required bit_synth CSV not found: {DATASET_CSV}. Upload the completed artifact to this exact Kaggle input path."
    )
print(f"Using prebuilt bit_synth CSV: {DATASET_CSV}")
if os.path.exists(MANIFEST_JSON):
    print(f"Using bit_synth manifest: {MANIFEST_JSON}")
else:
    MANIFEST_JSON = None
    print("bit_synth manifest not found at the fixed path; continuing without it.")

CSV_SCHEMA = {
    'id': pl.Utf8,
    'prompt': pl.Utf8,
    'answer': pl.Utf8,
    'generated_cot': pl.Utf8,
    'label': pl.Utf8,
    'assistant_style': pl.Utf8,
    'source_selection_tier': pl.Utf8,
}
train_df = pl.read_csv(DATASET_CSV, schema_overrides=CSV_SCHEMA)
if len(train_df) != SUBSAMPLE_SIZE:
    raise ValueError(f"Expected {SUBSAMPLE_SIZE} rows in {DATASET_CSV}, found {len(train_df)}")
if set(train_df.columns) != set(OUTPUT_COLUMNS):
    raise ValueError(f"Unexpected dataset columns: {train_df.columns}")
print(train_df.select(["label", "assistant_style"]).group_by(["label", "assistant_style"]).len().sort(["label", "assistant_style"]))
if MANIFEST_JSON is not None:
    manifest = json.load(open(MANIFEST_JSON, "r", encoding="utf-8"))
    manifest_preview = {
        "generated_rows": manifest.get("generated_rows"),
        "distinct_exact_rules_used": manifest.get("distinct_exact_rules_used"),
        "assistant_style": manifest.get("assistant_style"),
        "source_selection_tier": manifest.get("source_selection_tier"),
        "group_counts": manifest.get("group_counts"),
    }
    print(json.dumps(manifest_preview, ensure_ascii=False, indent=2))
train_df.head(3)


### 💬 Assistant-Only Prompt Construction

- User message stays README-compatible
- Assistant completion is either:
  - `trace_boxed`: verified reasoning trace + boxed answer
  - `boxed_only`: final boxed answer only
- Labels are masked so only assistant tokens contribute to the loss

✅ This remains the core assistant-only training change


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def apply_chat_template_safe(messages, *, add_generation_prompt, enable_thinking):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception:
        rendered = []
        for message in messages:
            rendered.append(f"<|{message['role']}|>\n{message['content']}")
        if add_generation_prompt:
            rendered.append("<|assistant|>\n<think>")
        return "\n".join(rendered)


def build_user_message(prompt: str) -> str:
    return prompt + "\n" + BOXED_INSTRUCTION


def render_assistant_message(example: dict[str, str]) -> str:
    if example["assistant_style"] == "trace_boxed":
        return f"{example['generated_cot']}\n\n\boxed{{{example['answer']}}}"
    if example["assistant_style"] == "boxed_only":
        return f"\boxed{{{example['answer']}}}"
    raise ValueError(f"Unsupported assistant_style: {example['assistant_style']}")


def tokenize_training_row(example: dict[str, str]) -> dict[str, Any]:
    user_message = build_user_message(example["prompt"])
    assistant_message = render_assistant_message(example)
    full_text = apply_chat_template_safe(
        [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        add_generation_prompt=False,
        enable_thinking=True,
    )
    assistant_char_start = full_text.find(assistant_message)
    if assistant_char_start < 0:
        raise ValueError(f"Assistant span not found in rendered chat for row {example['id']}")
    try:
        full_encoded = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True,
        )
    except (NotImplementedError, TypeError):
        full_encoded = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
    input_ids = list(full_encoded["input_ids"])
    attention_mask = list(full_encoded["attention_mask"])
    offset_mapping = full_encoded.get("offset_mapping")
    assistant_token_start = None
    if offset_mapping:
        for index, (start, _end) in enumerate(offset_mapping):
            if start >= assistant_char_start:
                assistant_token_start = index
                break
    if assistant_token_start is None:
        prefix_text = full_text[:assistant_char_start]
        prefix_ids = tokenizer(
            prefix_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )["input_ids"]
        assistant_token_start = len(prefix_ids)
    if assistant_token_start >= len(input_ids):
        raise ValueError(f"Assistant tokens were truncated out for row {example['id']}")
    labels = [-100] * assistant_token_start + input_ids[assistant_token_start:]
    return {
        "id": example["id"],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

hf_source = Dataset.from_pandas(train_df.to_pandas())
train_dataset = hf_source.map(tokenize_training_row, remove_columns=hf_source.column_names)
print(train_dataset[0].keys())
print(f"Rows tokenized: {len(train_dataset)}")


### 🧠 Model Loading & bit_synth Exact-Trace LoRA Training

- Keeps the baseline LoRA config (`rank=32`, `all-linear`, `dropout=0.05`)
- Replaces only the dataset artifact and sample count
- Trains a bit-manipulation exact-trace adapter aligned with README boxed-first evaluation

✅ Minimizes changes outside the data specialization step


In [ ]:
class AssistantOnlyDataCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(feature["input_ids"]) for feature in features)
        input_ids_batch = []
        attention_batch = []
        labels_batch = []
        for feature in features:
            pad = max_len - len(feature["input_ids"])
            input_ids_batch.append(feature["input_ids"] + [self.pad_token_id] * pad)
            attention_batch.append(feature["attention_mask"] + [0] * pad)
            labels_batch.append(feature["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long),
        }

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

import inspect

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "data_collator": AssistantOnlyDataCollator(tokenizer.pad_token_id),
}
trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)

print("Starting bit_synth exact-trace LoRA training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")


### 📦 Create Submission ZIP

- Same final packaging step as the baseline notebook
- Verifies `adapter_config.json` and `adapter_model.safetensors`

✅ Ready for Kaggle submission


In [ ]:
zip_path = "/kaggle/working/submission.zip"
print(f"Packaging files from {OUTPUT_DIR}...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")
print("✅ submission.zip is ready! You can now submit this file to the competition.")
